# V7 rate-recovery từ checkpoint −6,687%

Giữ nguyên VideoSwinLite, khởi tạo từ `best_task_bd_rate.pt` và không tiếp tục optimizer/dual state của epoch 15. Notebook tạo controller mới 800 video từ phần train pool chưa dùng, hiệu chỉnh proxy bằng H.264 thật, fine-tune 5 epoch với target BPP ratio 0.95 và đánh giá H.264 thật tại 7 QP trên full validation.

Bật GPU và Internet. Add Input: (1) Output đầy đủ chứa checkpoint −6,687% cùng proxy `best.pt` được checkpoint ghi nhận, (2) `checking` có `v5_fixed_split/split_manifest.json`, (3) `kineticscleaned`. Không cần codec cache. Run All theo thứ tự. Các checkpoint cũ chỉ được đọc và không bị ghi đè.


In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/proxy_v3")
if PROJECT.exists():
    subprocess.run(
        ["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", "main"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "-q", "--branch", "main",
         "https://github.com/munnn01/proxy_v3.git", str(PROJECT)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT / "requirements.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT))
print("Project:", PROJECT)


In [ ]:
import importlib
from kaggle_cells import v7_rate_recovery as recovery

recovery = importlib.reload(recovery)
RUN = recovery.prepare()


In [ ]:
CALIBRATED_PROXY = recovery.calibrate_proxy(RUN)
print("Calibrated proxy:", CALIBRATED_PROXY)


In [ ]:
CANDIDATE_PT, IS_FEASIBLE = recovery.fine_tune(RUN, CALIBRATED_PROXY)
print("Candidate:", CANDIDATE_PT)
print("Feasible on the new controller:", IS_FEASIBLE)


In [ ]:
EVAL_DIR = recovery.evaluate_candidate(RUN, CANDIDATE_PT)
print("Full 7-QP evaluation:", EVAL_DIR)


In [ ]:
from IPython.display import display, Image, FileLink

display(Image(filename=str(EVAL_DIR / "h264_top1_bpp_bd_rate.png")))
for path in [
    RUN.root / "selection.json",
    RUN.root / "recovery_split.json",
    EVAL_DIR / "metrics.csv",
    EVAL_DIR / "bd_rate.json",
    EVAL_DIR / "selection.json",
    EVAL_DIR / "per_video_metrics.csv",
]:
    display(FileLink(str(path)))
